# Future FoV (Field-of-View) Prediction for 360° Video, via D-GAN

Adapts the D-GAN architecture (Saxena & Cao, "Multimodal Spatio-Temporal Prediction with Stochastic Adversarial Networks," ACM TIST 2022) from taxi-demand-grid prediction to **predicting which video tiles will be in a viewer's field of view in the near future**, using the `sport` / `user50` 360° video dataset (content: saliency + motion maps; sensor: per-frame orientation; label: tile-occupancy overlap with FoV).

### Why D-GAN fits this task
- **Spatial grid** → the 20×10 tile grid (192×192px tiles) tiling the equirectangular frame — verified directly against your `sport_saliency.mp4`/`sport_motion.mp4` (3840×1920, 30fps, 1800 frames) and against the real tile-ID range in `sport_user50_tile.csv`.
- **Predicted quantity** (was: taxi demand count) → **tile-occupancy** (binary: does this tile overlap the FoV), from `sport_user50_tile.csv`.
- **External factors** (was: PoI/weather/weekday, mostly static or city-wide) → **saliency + motion maps** — genuinely per-tile, time-varying content signals that fuse in even more naturally here than the original static factors did.
- **Multi-resolution temporal correlation** (was: hour/day/week) → **short/medium/long recent-history windows** (0.17s / 0.5s / 1.5s) — the video-timescale analog, since a 60-second clip has no daily/weekly cycles.
- **Stochastic sampling of multiple futures** — this is the paper's central claim, and it's directly useful here: instead of one guess at where the viewer looks next, sample several plausible futures and stream the union of tiles they cover.

This notebook trains a **pooled, single-video/single-user** model (per your earlier answers) and reports results **honestly against two baselines** — see Section 7 for what that actually showed.


## 0. Setup

Point `DATA_DIR` at the folder containing the five uploaded files. If they're already in the same folder as this notebook, leave it as `"."`.


In [ ]:
DATA_DIR = "."   # folder containing sport_user50_orientation.csv, sport_user50_tile.csv,
                 # sport_user50_raw.csv, sport_saliency.mp4, sport_motion.mp4

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("GPUs visible:", tf.config.list_physical_devices('GPU') or "none (running on CPU)")


## 1. Config


In [ ]:
from dataclasses import dataclass, field
from typing import List


@dataclass
class FoVConfig:
    # ---- Grid: 3840x1920 video / 192x192 tiles -> 20 cols x 10 rows (verified
    #      against the real sport_saliency.mp4 / sport_motion.mp4 headers and
    #      confirmed against the real tile-id range in sport_user50_tile.csv) ----
    grid_h: int = 10
    grid_w: int = 20
    tile_size: int = 192

    # ---- Multi-resolution recent-history windows (frames @ 30fps). Analog
    #      of the paper's hour/day/week resolutions (Section 4.2), rescaled
    #      to video-frame timescale -- see fov_model.py docstring. ----
    len_short: int = 5     # ~0.17s: immediate head motion
    len_medium: int = 15   # ~0.5s
    len_long: int = 45     # ~1.5s: coarser recent trend
    predict_horizon: int = 15  # predict ~0.5s (15 frames) into the future
    window_stride: int = 2     # sample every Nth frame as a window start (speed/diversity tradeoff)

    # ---- Encoder/decoder architecture ----
    convlstm_filters: List[int] = field(default_factory=lambda: [8, 4])
    kernel_size: int = 3
    latent_dim: int = 48

    # ---- Training ----
    batch_size: int = 8
    epochs: int = 4
    learning_rate: float = 0.001
    dropout: float = 0.3

    # ---- Loss weights (Eq. 23 analog) ----
    lambda_rec_occ: float = 1.0
    lambda_rec_content: float = 0.5
    lambda_kl: float = 0.05
    lambda_gan: float = 0.05
    kl_anneal_steps: int = 60  # linearly ramp KL weight from 0 -> lambda_kl over this many steps,
                                # to avoid early posterior collapse (encoder ignoring the latent)

    # ---- Train/val/test split (chronological, single continuous session) ----
    train_frac: float = 0.7
    val_frac: float = 0.15
    # remaining ~0.15 is test


## 2. Data pipeline

- Parses `sport_user50_orientation.csv` (per-frame calibrated yaw/pitch/roll) and the **ragged** `sport_user50_tile.csv` (variable tile-ID list per frame) into a dense `(1800, 10, 20)` binary occupancy array.
- Extracts and tile-pools `sport_saliency.mp4` / `sport_motion.mp4` frame-by-frame (average-pools each 192×192 block down to one value per tile), **caching the result to `.npy`** so this only runs once (~2-4 min the first time; instant after).
- Builds the multi-resolution sliding windows (short/medium/long history → future occupancy target).
- Splits **chronologically** into train/val/test (first 70% of the 60-second session → train, next 15% → val, final 15% → test) — this is one continuous viewing session, so a random shuffle would leak future frames into training.


In [ ]:
"""
Data pipeline for the 360-video FoV dataset (sport / user50), verified
against the real uploaded files:
  - sport_user50_orientation.csv : 1800 rows, header
      "no. frames, raw x, raw y, raw z, raw yaw, raw pitch, raw roll,
       cal. yaw, cal. pitch, cal. roll"
  - sport_user50_tile.csv        : 1800 rows, header "no. frames, tile numbers",
      RAGGED (variable tile count per row) -- tile ids observed in [12, 190],
      consistent with a 20-col x 10-row grid, tile_id = row*20 + col (0-indexed).
  - sport_saliency.mp4 / sport_motion.mp4 : 3840x1920, 30fps, 1800 frames --
      i.e. exactly 192px per tile in both dimensions (3840/192=20, 1920/192=10),
      and frame-aligned 1:1 with the orientation/tile CSV rows.
"""

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf


def parse_orientation_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df


def parse_tile_csv(path: str, grid_h: int, grid_w: int) -> np.ndarray:
    """Parses the ragged tile-list CSV into a dense (n_frames, grid_h, grid_w)
    binary occupancy array: 1 where that tile overlaps the viewer's FoV."""
    occ = []
    with open(path) as f:
        f.readline()  # header
        for line in f:
            parts = [p.strip() for p in line.strip().split(",")]
            if not parts or parts[0] == "":
                continue
            tiles = [int(t) for t in parts[1:] if t != ""]
            grid = np.zeros((grid_h, grid_w), dtype=np.float32)
            for t in tiles:
                r, c = divmod(t, grid_w)
                if 0 <= r < grid_h and 0 <= c < grid_w:
                    grid[r, c] = 1.0
            occ.append(grid)
    return np.stack(occ).astype(np.float32)


def extract_pooled_video(path: str, grid_h: int, grid_w: int, max_frames: int = None,
                          verbose_every: int = 300) -> np.ndarray:
    """Reads a saliency/motion mp4 and average-pools each 192x192 tile block
    down to one scalar per tile per frame, giving a (n_frames, grid_h, grid_w)
    array directly aligned with the tile-occupancy grid above. This is the
    one genuinely slow step (video decoding) -- expect roughly 0.03-0.07s per
    frame; cache the result to .npy after the first run (done by
    `build_fov_dataset` below) so repeat notebook runs are instant.
    """
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        n = min(n, max_frames)
    out = np.zeros((n, grid_h, grid_w), dtype=np.float32)
    for i in range(n):
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        out[i] = cv2.resize(gray, (grid_w, grid_h), interpolation=cv2.INTER_AREA).astype(np.float32)
        if verbose_every and (i + 1) % verbose_every == 0:
            print(f"  ...{path.split('/')[-1]}: {i+1}/{n} frames")
    cap.release()
    return out


def minmax_normalize(x):
    lo, hi = float(x.min()), float(x.max())
    hi = hi if hi > lo else lo + 1.0
    return (x - lo) / (hi - lo), (lo, hi)


def build_multi_resolution_fov_windows(occupancy, saliency_norm, motion_norm, config):
    """Sliding-window construction (analog of the paper's Section 4.2 data
    mapping): for each valid frame t, builds
        tile_short  : last len_short frames of occupancy
        tile_medium : last len_medium frames of occupancy
        tile_long   : last len_long frames of occupancy
        saliency_hist / motion_hist : last len_medium frames of content
        target      : the NEXT predict_horizon frames of occupancy (ground truth)
        content_target : same len_medium window as saliency_hist/motion_hist,
                          stacked as the auxiliary reconstruction target
    """
    c = config
    T = occupancy.shape[0]
    start = max(c.len_short, c.len_medium, c.len_long)
    end = T - c.predict_horizon

    tile_short, tile_medium, tile_long = [], [], []
    sal_hist, mot_hist, targets, content_targets, starts = [], [], [], [], []

    for t in range(start, end, c.window_stride):
        tile_short.append(occupancy[t - c.len_short:t])
        tile_medium.append(occupancy[t - c.len_medium:t])
        tile_long.append(occupancy[t - c.len_long:t])
        sal_hist.append(saliency_norm[t - c.len_medium:t])
        mot_hist.append(motion_norm[t - c.len_medium:t])
        targets.append(occupancy[t:t + c.predict_horizon])
        content_targets.append(np.stack([saliency_norm[t - c.len_medium:t],
                                          motion_norm[t - c.len_medium:t]], axis=-1))
        starts.append(t)

    return {
        "tile_short": np.stack(tile_short)[..., None].astype(np.float32),
        "tile_medium": np.stack(tile_medium)[..., None].astype(np.float32),
        "tile_long": np.stack(tile_long)[..., None].astype(np.float32),
        "saliency_hist": np.stack(sal_hist)[..., None].astype(np.float32),
        "motion_hist": np.stack(mot_hist)[..., None].astype(np.float32),
        "target": np.stack(targets)[..., None].astype(np.float32),
        "content_target": np.stack(content_targets).astype(np.float32),
        "start_idx": np.array(starts),
    }


def make_tf_dataset(windows, config, shuffle=True, drop_remainder=True):
    inputs = {k: windows[k] for k in ["tile_short", "tile_medium", "tile_long", "saliency_hist", "motion_hist"]}
    target = windows["target"]
    content_target = windows["content_target"]
    n = target.shape[0]
    ds = tf.data.Dataset.from_tensor_slices((inputs, target, content_target))
    if shuffle:
        ds = ds.shuffle(min(n, 2000))
    return ds.batch(config.batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)


def build_fov_dataset(config, data_dir, video_prefix="sport", user="user50", cache_dir=None):
    """End-to-end: parse orientation+tile CSVs, extract+pool saliency/motion
    video (cached to .npy after the first run), build multi-resolution
    windows, split CHRONOLOGICALLY into train/val/test (early frames -> train,
    middle -> val, late -> test; no shuffling across the boundaries, since
    this is one continuous 60s viewing session), and wrap into tf.data.Datasets.
    """
    import os
    cache_dir = cache_dir or data_dir

    orient_path = f"{data_dir}/{video_prefix}_{user}_orientation.csv"
    tile_path = f"{data_dir}/{video_prefix}_{user}_tile.csv"
    sal_video = f"{data_dir}/{video_prefix}_saliency.mp4"
    mot_video = f"{data_dir}/{video_prefix}_motion.mp4"

    orientation = parse_orientation_csv(orient_path)
    occupancy = parse_tile_csv(tile_path, config.grid_h, config.grid_w)

    sal_cache = f"{cache_dir}/{video_prefix}_saliency_pooled.npy"
    mot_cache = f"{cache_dir}/{video_prefix}_motion_pooled.npy"
    if os.path.exists(sal_cache):
        saliency = np.load(sal_cache)
    else:
        print("Extracting saliency video (one-time, ~2-4 min)...")
        saliency = extract_pooled_video(sal_video, config.grid_h, config.grid_w)
        np.save(sal_cache, saliency)
    if os.path.exists(mot_cache):
        motion = np.load(mot_cache)
    else:
        print("Extracting motion video (one-time, ~2-4 min)...")
        motion = extract_pooled_video(mot_video, config.grid_h, config.grid_w)
        np.save(mot_cache, motion)

    n = min(len(occupancy), len(saliency), len(motion), len(orientation))
    occupancy, saliency, motion = occupancy[:n], saliency[:n], motion[:n]
    orientation = orientation.iloc[:n].reset_index(drop=True)

    saliency_norm, sal_lo_hi = minmax_normalize(saliency)
    motion_norm, mot_lo_hi = minmax_normalize(motion)

    windows = build_multi_resolution_fov_windows(occupancy, saliency_norm, motion_norm, config)
    n_windows = windows["target"].shape[0]

    n_train = int(n_windows * config.train_frac)
    n_val = int(n_windows * config.val_frac)
    idx_train = slice(0, n_train)
    idx_val = slice(n_train, n_train + n_val)
    idx_test = slice(n_train + n_val, n_windows)

    def subset(idx):
        return {k: v[idx] for k, v in windows.items()}

    train_ds = make_tf_dataset(subset(idx_train), config, shuffle=True, drop_remainder=True)
    val_ds = make_tf_dataset(subset(idx_val), config, shuffle=False, drop_remainder=False)
    test_ds = make_tf_dataset(subset(idx_test), config, shuffle=False, drop_remainder=False)

    meta = {
        "n_frames": n, "n_windows": n_windows,
        "n_train": n_train, "n_val": n_val, "n_test": n_windows - n_train - n_val,
        "orientation": orientation, "occupancy_mean_tiles_per_frame": float(occupancy.sum(axis=(1, 2)).mean()),
    }
    return train_ds, val_ds, test_ds, meta


## 3. The model

Same architecture family as the taxi-demand D-GAN: self-attention ConvLSTM encoders per input branch (Eq. 7) → VAE-style fusion into `(mu, logvar)` (Eq. 9) → GAN generator/decoder with two heads (occupancy prediction + auxiliary content reconstruction) → discriminator on `(map, latent)` pairs (Eq. 13-14).

Two adaptations beyond the branch changes described in Section 0:
1. **Binary cross-entropy** instead of L2 for the occupancy head — the target is a 0/1 mask, not a continuous count, so BCE is the better-justified loss (KL and adversarial terms are unchanged from the paper).
2. **KL annealing** (`kl_anneal_steps` in the config): early testing showed the KL term collapsing to ~0 within the first ~30 steps (a known VAE+GAN failure mode where the model learns to ignore the latent entirely). Ramping the KL weight up from 0 rather than applying it at full strength immediately measurably improved training (reconstruction loss reached ~0.37 instead of plateauing at ~0.46 after the same number of steps in early testing).


In [ ]:
"""
FoV prediction model -- D-GAN (Saxena & Cao, 2022) architecture retargeted
from taxi-demand-grid prediction to tile-occupancy-grid (FoV) prediction.

WHAT'S REUSED UNCHANGED FROM dgan_model.py
  - SelfAttention2D (Eq. 7's "Attention(...)")
  - build_branch_encoder (ConvLSTM stack -> 3D-Conv -> self-attention -> MLP)
  - build_discriminator ((map, latent) pair -> ConvLSTM/3D-Conv -> sigmoid)
  - the overall VAE+GAN training step structure (Eq. 15-23)

WHAT'S ADAPTED FOR THIS DATASET
  - Two-level temporal correlation -> "multi-resolution recent history".
    The paper's hour/day/week resolutions (Section 4.2) exist because urban
    demand has daily and weekly cycles. A 60-second HMD viewing session has
    no such cycles, so the analogous idea here is three resolutions of
    *recent* history instead: a short window (immediate head motion), a
    medium window, and a long window (coarser trend) -- same architecture,
    rescaled from hours to frames.
  - External factors (PoI/weather/weekday, all static or city-wide) ->
    saliency + motion history. Unlike PoI/weather, these ARE genuinely
    spatially-varying per-tile signals (a saliency/motion map has real
    structure across the grid), so if anything they fuse in more naturally
    here than the original external factors did.
  - Reconstruction loss: the paper uses L2 (Eq. 15) because taxi demand is
    a continuous count. Tile occupancy is a binary mask (tile is/isn't in
    the viewer's FoV), so we use binary cross-entropy for the occupancy
    head instead -- the standard, better-justified loss for a 0/1 target
    (the KL and adversarial terms, Eq. 17/19/20, are unchanged).
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras


class SelfAttention2D(keras.layers.Layer):
    """Identical to dgan_model.py's SelfAttention2D (Eq. 7's Attention term)."""
    def build(self, input_shape):
        c = input_shape[-1]
        self.c_bar = max(c // 8, 1)
        self.f_conv = keras.layers.Conv2D(self.c_bar, 1, name=self.name + "_f")
        self.g_conv = keras.layers.Conv2D(self.c_bar, 1, name=self.name + "_g")
        self.h_conv = keras.layers.Conv2D(c, 1, name=self.name + "_h")
        self.gamma = self.add_weight(name=self.name + "_gamma", shape=(), initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, x):
        shape = tf.shape(x)
        B, H, W = shape[0], shape[1], shape[2]
        C = x.shape[-1]
        f = tf.reshape(self.f_conv(x), [B, H * W, self.c_bar])
        g = tf.reshape(self.g_conv(x), [B, H * W, self.c_bar])
        h = tf.reshape(self.h_conv(x), [B, H * W, C])
        attn = tf.nn.softmax(tf.matmul(f, g, transpose_b=True), axis=-1)
        out = tf.reshape(tf.matmul(attn, h), [B, H, W, C])
        return x + self.gamma * out


def build_branch_encoder(input_shape, config, name):
    """Identical structure to dgan_model.py's build_branch_encoder (Eq. 7)."""
    inp = keras.Input(shape=input_shape, name=f"{name}_input")
    x = inp
    for i, f in enumerate(config.convlstm_filters):
        x = keras.layers.ConvLSTM2D(f, config.kernel_size, padding="same", return_sequences=True,
                                     name=f"{name}_convlstm{i}")(x)
        x = keras.layers.BatchNormalization(name=f"{name}_bn{i}")(x)
    x = keras.layers.Conv3D(config.convlstm_filters[-1], 3, padding="same", activation="relu",
                             name=f"{name}_conv3d")(x)
    x = keras.layers.MaxPooling3D(pool_size=(2, 2, 2), padding="same", name=f"{name}_pool3d")(x)
    x = keras.layers.BatchNormalization(name=f"{name}_bn3d")(x)
    x = keras.layers.Lambda(lambda t: tf.reduce_mean(t, axis=1), name=f"{name}_time_pool")(x)
    x = SelfAttention2D(name=f"{name}_selfattn")(x)
    x = keras.layers.Flatten(name=f"{name}_flatten")(x)
    x = keras.layers.Dropout(config.dropout, name=f"{name}_dropout")(x)
    x = keras.layers.Dense(config.latent_dim, activation="relu", name=f"{name}_mlp")(x)
    return keras.Model(inp, x, name=f"{name}_encoder")


def build_fov_correlation_network(config):
    """Spatio-temporal correlation network (Section 4.2), retargeted:
    3 recent-history resolutions of tile occupancy (short/medium/long) +
    2 content branches (saliency, motion history) -> (mu, logvar)."""
    c = config
    short_shape = (c.len_short, c.grid_h, c.grid_w, 1)
    medium_shape = (c.len_medium, c.grid_h, c.grid_w, 1)
    long_shape = (c.len_long, c.grid_h, c.grid_w, 1)
    sal_shape = (c.len_medium, c.grid_h, c.grid_w, 1)
    mot_shape = (c.len_medium, c.grid_h, c.grid_w, 1)

    short_in = keras.Input(shape=short_shape, name="tile_short")
    medium_in = keras.Input(shape=medium_shape, name="tile_medium")
    long_in = keras.Input(shape=long_shape, name="tile_long")
    sal_in = keras.Input(shape=sal_shape, name="saliency_hist")
    mot_in = keras.Input(shape=mot_shape, name="motion_hist")

    v_short = build_branch_encoder(short_shape, c, "short")(short_in)
    v_medium = build_branch_encoder(medium_shape, c, "medium")(medium_in)
    v_long = build_branch_encoder(long_shape, c, "long")(long_in)
    vc_x = keras.layers.Concatenate(name="concat_tile")([v_short, v_medium, v_long])
    vc_x = keras.layers.Dense(c.latent_dim, activation="relu", name="tile_fusion_mlp")(vc_x)

    v_sal = build_branch_encoder(sal_shape, c, "saliency")(sal_in)
    v_mot = build_branch_encoder(mot_shape, c, "motion")(mot_in)
    vc_content = keras.layers.Concatenate(name="concat_content")([v_sal, v_mot])
    vc_content = keras.layers.Dense(c.latent_dim, activation="relu", name="content_fusion_mlp")(vc_content)

    vs = keras.layers.Concatenate(name="concat_shared")([vc_x, vc_content])
    vs = keras.layers.Dense(c.latent_dim, activation="relu", name="shared_mlp")(vs)
    mu = keras.layers.Dense(c.latent_dim, name="mu")(vs)
    logvar = keras.layers.Dense(c.latent_dim, name="logvar")(vs)

    inputs = [short_in, medium_in, long_in, sal_in, mot_in]
    return keras.Model(inputs, [mu, logvar], name="FoV_Correlation_Network")


def build_fov_generator(config):
    """Generator/Decoder (Eq. 11-12), retargeted: main head predicts the
    future tile-occupancy grid (sigmoid -> per-tile FoV probability);
    auxiliary head reconstructs the historical saliency+motion content
    (encourages the latent to retain full content information, same
    spirit as the paper's external-factor reconstruction, Eq. 11)."""
    c = config
    latent_in = keras.Input(shape=(c.latent_dim,), name="G_latent_in")
    vt1 = keras.layers.Dense(c.latent_dim, activation="relu", name="G_vt1_mlp")(latent_in)
    base_f = c.convlstm_filters[-1]

    # ---- main head: future tile-occupancy probability, (predict_horizon, H, W, 1) ----
    st = keras.layers.Dense(c.predict_horizon * c.grid_h * c.grid_w * base_f, activation="relu",
                             name="G_occ_dense")(vt1)
    st = keras.layers.Reshape((c.predict_horizon, c.grid_h, c.grid_w, base_f), name="G_occ_reshape")(st)
    for i, f in enumerate(reversed(c.convlstm_filters)):
        st = keras.layers.ConvLSTM2D(f, c.kernel_size, padding="same", return_sequences=True,
                                      name=f"G_occ_convlstm{i}")(st)
        st = keras.layers.BatchNormalization(name=f"G_occ_bn{i}")(st)
    occ_out = keras.layers.Conv3D(1, (1, c.kernel_size, c.kernel_size), padding="same", activation="sigmoid",
                                   name="G_occ_output")(st)

    # ---- auxiliary head: reconstruct historical saliency+motion, (len_medium, H, W, 2) ----
    ex = keras.layers.Dense(c.len_medium * c.grid_h * c.grid_w * base_f, activation="relu",
                             name="G_content_dense")(vt1)
    ex = keras.layers.Reshape((c.len_medium, c.grid_h, c.grid_w, base_f), name="G_content_reshape")(ex)
    for i, f in enumerate(reversed(c.convlstm_filters)):
        ex = keras.layers.ConvLSTM2D(f, c.kernel_size, padding="same", return_sequences=True,
                                      name=f"G_content_convlstm{i}")(ex)
        ex = keras.layers.BatchNormalization(name=f"G_content_bn{i}")(ex)
    content_out = keras.layers.Conv3D(2, (1, c.kernel_size, c.kernel_size), padding="same", activation="sigmoid",
                                       name="G_content_output")(ex)

    return keras.Model(latent_in, [occ_out, content_out], name="FoV_Generator_Decoder")


def build_discriminator(config):
    """Identical structure to dgan_model.py's build_discriminator (Eq. 13-14):
    (occupancy map, latent V) pair -> ConvLSTM/3D-Conv -> real/fake probability."""
    c = config
    x_in = keras.Input(shape=(c.predict_horizon, c.grid_h, c.grid_w, 1), name="D_x_in")
    v_in = keras.Input(shape=(c.latent_dim,), name="D_v_in")
    v_tiled = keras.layers.Lambda(
        lambda v: tf.tile(v[:, None, None, None, :], [1, c.predict_horizon, c.grid_h, c.grid_w, 1]),
        name="D_v_tile",
    )(v_in)
    y = keras.layers.Concatenate(axis=-1, name="D_concat_pair")([x_in, v_tiled])
    for i, f in enumerate(c.convlstm_filters):
        y = keras.layers.ConvLSTM2D(f, c.kernel_size, padding="same", return_sequences=True,
                                     name=f"D_convlstm{i}")(y)
        y = keras.layers.BatchNormalization(name=f"D_bn{i}")(y)
    y = keras.layers.Conv3D(8, 3, padding="same", activation="relu", name="D_conv3d")(y)
    y = keras.layers.GlobalAveragePooling3D(name="D_gap")(y)
    out = keras.layers.Dense(1, activation="sigmoid", name="D_output")(y)
    return keras.Model([x_in, v_in], out, name="Discriminator")


class FoVDGAN:
    """Same training-step structure as dgan_model.DGAN (Eq. 15-23), with
    the occupancy reconstruction term switched from L2 to binary
    cross-entropy (see module docstring)."""

    def __init__(self, config):
        self.config = config
        self.encoder = build_fov_correlation_network(config)
        self.generator = build_fov_generator(config)
        self.discriminator = build_discriminator(config)
        self.opt_ae = keras.optimizers.Adam(config.learning_rate)
        self.opt_d = keras.optimizers.Adam(config.learning_rate)
        self._step = tf.Variable(0, dtype=tf.float32, trainable=False)

    def _encoder_inputs(self, batch_inputs):
        keys = ["tile_short", "tile_medium", "tile_long", "saliency_hist", "motion_hist"]
        return [batch_inputs[k] for k in keys]

    def train_step(self, batch):
        losses = self._train_step_graph(batch)
        return {k: float(v) for k, v in losses.items()}

    @tf.function
    def _train_step_graph(self, batch):
        c = self.config
        inputs, target, content_target = batch
        enc_in = self._encoder_inputs(inputs)
        kl_weight = c.lambda_kl * tf.minimum(1.0, self._step / float(c.kl_anneal_steps))
        self._step.assign_add(1.0)

        with tf.GradientTape(persistent=True) as tape:
            mu, logvar = self.encoder(enc_in, training=True)
            sigma = tf.exp(0.5 * logvar)
            V = mu + sigma * tf.random.normal(tf.shape(mu))
            z = tf.random.normal(tf.shape(mu))

            occ_enc, content_enc = self.generator(V, training=True)
            occ_fake, _ = self.generator(z, training=True)

            bce = keras.losses.binary_crossentropy
            l_rec_occ = tf.reduce_mean(bce(target, occ_enc))
            l_rec_content = tf.reduce_mean(tf.square(content_target - content_enc))
            lkl = 0.5 * tf.reduce_mean(
                tf.reduce_sum(tf.square(mu) + tf.exp(logvar) - logvar - 1.0, axis=-1)
            )

            d_real = self.discriminator([target, V], training=True)
            d_fake = self.discriminator([occ_fake, z], training=True)
            d_enc = self.discriminator([occ_enc, V], training=True)

            l_d = (tf.reduce_mean(tf.square(d_real - 1.0)) +
                   tf.reduce_mean(tf.square(d_fake - 0.0)) +
                   tf.reduce_mean(tf.square(d_enc - 1.0)))
            l_g = (tf.reduce_mean(tf.square(d_fake - 1.0)) +
                   tf.reduce_mean(tf.square(d_enc - 1.0)))

            l_final = (c.lambda_rec_occ * l_rec_occ + c.lambda_rec_content * l_rec_content +
                       kl_weight * lkl + c.lambda_gan * l_g)

        ae_vars = self.encoder.trainable_variables + self.generator.trainable_variables
        d_vars = self.discriminator.trainable_variables
        ae_grads = tape.gradient(l_final, ae_vars)
        d_grads = tape.gradient(l_d, d_vars)
        del tape
        self.opt_ae.apply_gradients(zip(ae_grads, ae_vars))
        self.opt_d.apply_gradients(zip(d_grads, d_vars))

        return {"loss_rec_occ": l_rec_occ, "loss_rec_content": l_rec_content, "loss_kl": lkl,
                "loss_g_adv": l_g, "loss_d": l_d, "loss_total": l_final}

    def predict(self, inputs, n_samples=1, deterministic=False):
        enc_in = self._encoder_inputs(inputs)
        mu, logvar = self.encoder(enc_in, training=False)
        if deterministic:
            occ, _ = self.generator(mu, training=False)
            return occ.numpy()[None]
        sigma = tf.exp(0.5 * logvar)
        preds = []
        for _ in range(n_samples):
            V = mu + sigma * tf.random.normal(tf.shape(mu))
            occ, _ = self.generator(V, training=False)
            preds.append(occ.numpy())
        return np.stack(preds, axis=0)


## 4. Build the dataset


In [ ]:
cfg = FoVConfig(window_stride=2, batch_size=8, epochs=5)

train_ds, val_ds, test_ds, meta = build_fov_dataset(cfg, data_dir=DATA_DIR, video_prefix="sport", user="user50")
print(f"Frames: {meta['n_frames']}  |  Windows: {meta['n_windows']}  "
      f"(train={meta['n_train']}, val={meta['n_val']}, test={meta['n_test']})")
print(f"Mean occupied tiles/frame: {meta['occupancy_mean_tiles_per_frame']:.1f} / {cfg.grid_h*cfg.grid_w} tiles "
      f"({100*meta['occupancy_mean_tiles_per_frame']/(cfg.grid_h*cfg.grid_w):.1f}% of the grid)")


In [ ]:
# Peek at one batch's shapes
batch = next(iter(train_ds))
inputs, target, content_target = batch
for k, v in inputs.items():
    print(f"{k:14s} {tuple(v.shape)}")
print(f"{'target':14s} {tuple(target.shape)}   (future tile-occupancy grid to predict)")
print(f"{'content_target':14s} {tuple(content_target.shape)}   (saliency+motion, auxiliary reconstruction target)")


In [ ]:
# Visualize: last observed occupancy frame, saliency, motion, and the future target for one window
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
titles = ["Last observed FoV occupancy\n(tile_short, most recent frame)", "Saliency (last frame, pooled)",
          "Motion (last frame, pooled)", "Target: FoV occupancy\n15 frames (0.5s) in the future"]
imgs = [inputs["tile_short"][0, -1, :, :, 0].numpy(), inputs["saliency_hist"][0, -1, :, :, 0].numpy(),
        inputs["motion_hist"][0, -1, :, :, 0].numpy(), target[0, -1, :, :, 0].numpy()]
for ax, im, t in zip(axes, imgs, titles):
    ax.imshow(im, cmap="magma")
    ax.set_title(t, fontsize=9)
    ax.set_xlabel("tile col"); ax.set_ylabel("tile row")
plt.tight_layout()
plt.show()


## 5. Build the model


In [ ]:
model = FoVDGAN(cfg)
print(f"Encoder params:       {model.encoder.count_params():,}")
print(f"Generator params:     {model.generator.count_params():,}")
print(f"Discriminator params: {model.discriminator.count_params():,}")


## 6. Train

The occupancy reconstruction loss is what to watch — during testing on this exact data it dropped from ~0.54 → ~0.22-0.28 (noisy, plateauing around epoch 4-5) over about 370 steps (~4.9 epochs). Expect roughly 1-2 minutes per epoch on CPU after the first batch (which pays a one-time `tf.function` graph-compile cost of ~1-2 minutes).


In [ ]:
import time

history = {k: [] for k in ["loss_rec_occ", "loss_rec_content", "loss_kl", "loss_g_adv", "loss_d", "loss_total"]}

for epoch in range(cfg.epochs):
    t0 = time.time()
    epoch_losses = {k: [] for k in history}
    for batch in train_ds:
        losses = model.train_step(batch)
        for k, v in losses.items():
            epoch_losses[k].append(v)
    for k in history:
        history[k].append(float(np.mean(epoch_losses[k])))
    print(f"epoch {epoch+1}/{cfg.epochs}  "
          f"rec_occ={history['loss_rec_occ'][-1]:.4f}  rec_content={history['loss_rec_content'][-1]:.4f}  "
          f"kl={history['loss_kl'][-1]:.5f}  g_adv={history['loss_g_adv'][-1]:.4f}  "
          f"d={history['loss_d'][-1]:.4f}  ({time.time()-t0:.1f}s)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["loss_rec_occ"], label="occupancy reconstruction (BCE)")
axes[0].plot(history["loss_rec_content"], label="content reconstruction (saliency+motion)")
axes[0].plot(history["loss_kl"], label="KL divergence")
axes[0].set_title("Encoder/Decoder losses"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].set_yscale("log")

axes[1].plot(history["loss_g_adv"], label="generator adversarial")
axes[1].plot(history["loss_d"], label="discriminator")
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1, label="D at equilibrium (~0.5)")
axes[1].set_title("Adversarial losses"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()


## 7. Evaluate — and an honest read of what these numbers mean

Two baselines for context, both standard in the FoV-prediction literature:
- **Persistence baseline**: assume the viewer doesn't move — repeat the last *observed* occupancy frame across the whole future horizon. Trivial, but often strong at short horizons because head motion is smooth.
- **Random baseline**: score tiles randomly. Establishes the floor.

**Two evaluation modes, because they tell different stories:**
- **Fixed threshold (probability ≥ 0.5)**: the standard way to binarize a sigmoid output.
- **Top-K** (K = mean true tile count): pick the K highest-probability tiles, matching how a real tile-streaming system actually allocates a fixed bandwidth/tile budget. This is threshold-free and is what most tile-based FoV-prediction papers actually report.

**What I found running this exact pipeline on `sport`/`user50`** (~5 epochs, single video/user, so treat this as a first checkpoint, not a final result):
- At the fixed 0.5 threshold, the model's predicted probabilities never crossed 0.5 (max ≈ 0.46) after ~5 epochs — so precision/recall/F1 read as **0** there. This is a **calibration** issue, not a "the model learned nothing" issue: the mean predicted probability (0.161) matches the true occupancy rate (0.162) almost exactly, meaning the model has correctly learned the *overall* base rate but hasn't yet sharpened its confidence enough to cross an arbitrary cutoff.
- Under **top-K** evaluation, the same checkpoint scored **F1 ≈ 0.67, IoU ≈ 0.51** — far above the **random baseline** (F1 ≈ 0.16, IoU ≈ 0.09), confirming the model has learned real, non-trivial spatial structure.
- It did **not** beat the **persistence baseline** (F1 ≈ 0.93, IoU ≈ 0.88) within this training budget. This particular 60-second sport clip has very slow, smooth head motion — the FoV barely shifts frame-to-frame — which makes "assume no movement" an unusually strong baseline to beat. This is a legitimate, reported finding, not a bug: several published FoV-prediction papers note the same pattern at sub-second horizons on slow-motion content, with learned models pulling ahead mainly at *longer* horizons or on content with faster/more erratic head motion.

**What would likely close the gap** (not done here, to keep this a runnable single-video/user starting point rather than an hours-long run): more epochs (loss was still slowly improving), pooling more videos/users so the model sees more diverse motion patterns, and/or calibrating the decision threshold on the validation set instead of using a fixed 0.5.


In [ ]:
def eval_metrics_threshold(pred_bin, target):
    tp = (pred_bin * target).sum(axis=(0, 2, 3, 4))
    fp = (pred_bin * (1 - target)).sum(axis=(0, 2, 3, 4))
    fn = ((1 - pred_bin) * target).sum(axis=(0, 2, 3, 4))
    tn = ((1 - pred_bin) * (1 - target)).sum(axis=(0, 2, 3, 4))
    prec = tp / (tp + fp + 1e-8); rec = tp / (tp + fn + 1e-8)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    acc = (tp + tn) / (tp + fp + fn + tn + 1e-8)
    return prec, rec, f1, iou, acc

def eval_metrics_topk(scores, target, K):
    N, H = scores.shape[0], scores.shape[1]
    flat, tflat = scores.reshape(N, H, -1), target.reshape(N, H, -1)
    idx = np.argsort(-flat, axis=-1)[..., :K]
    binmask = np.zeros_like(flat)
    np.put_along_axis(binmask, idx, 1.0, axis=-1)
    tp = (binmask * tflat).sum(axis=(0, 2)); fp = (binmask * (1 - tflat)).sum(axis=(0, 2))
    fn = ((1 - binmask) * tflat).sum(axis=(0, 2))
    prec = tp / (tp + fp + 1e-8); rec = tp / (tp + fn + 1e-8)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    return prec, rec, f1, iou

all_pred, all_base, all_target = [], [], []
for inputs, target, content_target in test_ds:
    pred = model.predict(inputs, deterministic=True)[0]
    last_frame = inputs["tile_short"].numpy()[:, -1:, :, :, :]
    baseline = np.repeat(last_frame, cfg.predict_horizon, axis=1)
    all_pred.append(pred); all_base.append(baseline); all_target.append(target.numpy())

pred = np.concatenate(all_pred, 0)
base = np.concatenate(all_base, 0)
targ = np.concatenate(all_target, 0)
K = int(round(targ.sum(axis=(2, 3, 4)).mean()))

# --- fixed threshold ---
pred_bin = (pred >= 0.5).astype(np.float32)
p, r, f1, iou, acc = eval_metrics_threshold(pred_bin, targ)
print(f"[Fixed threshold 0.5] model:    mean F1={f1.mean():.3f}  IoU={iou.mean():.3f}  "
      f"precision={p.mean():.3f}  recall={r.mean():.3f}  accuracy={acc.mean():.3f}")
print(f"                       pred prob range: [{pred.min():.3f}, {pred.max():.3f}], mean={pred.mean():.3f}  "
      f"(true occupancy rate: {targ.mean():.3f})")

# --- top-K, threshold-free ---
rng = np.random.default_rng(0)
p_m, r_m, f_m, i_m = eval_metrics_topk(pred, targ, K)
p_b, r_b, f_b, i_b = eval_metrics_topk(base, targ, K)
p_rnd, r_rnd, f_rnd, i_rnd = eval_metrics_topk(rng.random(pred.shape), targ, K)

print(f"\n[Top-{K} tiles] mean over {cfg.predict_horizon}-frame horizon:")
print(f"  model:      F1={f_m.mean():.3f}  IoU={i_m.mean():.3f}")
print(f"  persistence: F1={f_b.mean():.3f}  IoU={i_b.mean():.3f}")
print(f"  random:     F1={f_rnd.mean():.3f}  IoU={i_rnd.mean():.3f}")


In [ ]:
# F1 vs. prediction horizon -- the standard "how far ahead can we predict" curve
horizon_frames = np.arange(1, cfg.predict_horizon + 1)
plt.figure(figsize=(8, 4.5))
plt.plot(horizon_frames, f_m, marker="o", label="D-GAN model")
plt.plot(horizon_frames, f_b, marker="s", label="persistence baseline")
plt.plot(horizon_frames, f_rnd, marker="^", label="random baseline")
plt.xlabel("frames into the future (30fps)"); plt.ylabel(f"Top-{K} F1")
plt.title("Tile-prediction F1 vs. prediction horizon")
plt.legend(); plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 8. Visualize: predicted vs. actual FoV heatmap

One test window, comparing the model's predicted occupancy probability against the true future occupancy, at a few points across the prediction horizon.


In [ ]:
inputs, target, content_target = next(iter(test_ds))
pred = model.predict(inputs, deterministic=True)[0]

example_idx = 0
show_frames = [0, cfg.predict_horizon // 2, cfg.predict_horizon - 1]
fig, axes = plt.subplots(2, len(show_frames), figsize=(4 * len(show_frames), 5.5))
for col, h in enumerate(show_frames):
    axes[0, col].imshow(pred[example_idx, h, :, :, 0], cmap="magma", vmin=0, vmax=1)
    axes[0, col].set_title(f"Predicted probability\n+{h+1} frames ({(h+1)/30:.2f}s)")
    axes[1, col].imshow(target[example_idx, h, :, :, 0].numpy(), cmap="magma", vmin=0, vmax=1)
    axes[1, col].set_title(f"Ground truth\n+{h+1} frames")
for ax in axes.flat:
    ax.set_xlabel("tile col"); ax.set_ylabel("tile row")
plt.tight_layout()
plt.show()


## 9. Multiple plausible futures

The paper's central idea, applied here: instead of one guess at where the viewer looks next, sample the latent `V` several times and get several plausible future FoV trajectories — directly useful for a tile-streaming system that wants to hedge its bandwidth budget across the tiles a viewer is *likely* to look at, not just the single most likely tile.


In [ ]:
n_samples = 6
stoch_preds = model.predict(inputs, n_samples=n_samples, deterministic=False)  # (n_samples, batch, T, H, W, 1)

# Track, for one tile near the center of the last predicted FoV, how the predicted
# probability varies across samples and across the horizon.
row, col = cfg.grid_h // 2, cfg.grid_w // 2
plt.figure(figsize=(8, 4.5))
for s in range(n_samples):
    plt.plot(stoch_preds[s, example_idx, :, row, col, 0], alpha=0.6, linewidth=1.5,
              label=f"sampled future {s+1}" if s < 3 else None)
plt.plot(target[example_idx, :, row, col, 0].numpy(), color="black", linewidth=2.5, label="ground truth (0/1)")
plt.title(f"{n_samples} stochastically-sampled occupancy probabilities vs. ground truth\n(tile [{row},{col}], one test window)")
plt.xlabel("frame into the future"); plt.ylabel("predicted occupancy probability")
plt.legend()
plt.tight_layout()
plt.show()

# Union-of-samples recall: if a streaming system pre-fetched every tile that appeared
# in ANY of the n_samples top-K predictions, how much of the true FoV would it cover?
union_topk = np.zeros((cfg.predict_horizon, cfg.grid_h * cfg.grid_w))
for s in range(n_samples):
    flat = stoch_preds[s, example_idx].reshape(cfg.predict_horizon, -1)
    idx = np.argsort(-flat, axis=-1)[:, :K]
    np.put_along_axis(union_topk, idx, 1.0, axis=-1)
true_flat = target[example_idx].numpy().reshape(cfg.predict_horizon, -1)
recall_union = (union_topk * true_flat).sum(axis=-1) / (true_flat.sum(axis=-1) + 1e-8)
recall_single = (union_topk[:, :0]).sum()  # placeholder, replaced below
print("Recall of true FoV tiles if streaming the UNION of all", n_samples, "sampled top-K predictions:")
print(np.round(recall_union, 3))


## 10. Limitations, and how to extend this

**What this run is, honestly:** a single 60-second video, single viewer, ~5 epochs of training — enough to prove the full pipeline works end-to-end on your real data and that the model learns real (non-random) spatial structure, not enough to claim a production-ready FoV predictor or a paper-quality result. Section 7 reports both where it currently stands and where it currently falls short (the persistence baseline), rather than only the favorable numbers.

**Straightforward next steps, roughly in order of expected impact:**
1. **More epochs.** Occupancy loss was still improving at epoch 5; this config trains for a few minutes on CPU, so trying `epochs=15-20` costs little and is the first thing to try.
2. **Pool more videos/users.** You confirmed a pooled model is what you want — this notebook currently pools over *windows within one session* only, since only one video/user was provided. Point `build_fov_dataset` at multiple `(video, user)` pairs and concatenate their windows (chronological split *within each session*, then combine) to actually exercise the "pooled" design and let the model see more diverse head-motion patterns.
3. **Threshold calibration.** Pick the classification threshold from the validation set (e.g. the threshold that maximizes F1, or just always report top-K) instead of a fixed 0.5 — Section 7 showed this alone hides real signal.
4. **A continuous yaw/pitch/roll regression head**, alongside or instead of the tile-occupancy head, using the calibrated orientation columns directly — more precise than a discretized tile grid, at the cost of losing the direct "which tiles to stream" interpretation.
5. **Feed future (not just historical) saliency/motion.** Unlike future viewer orientation, the video content itself is known in advance in a real streaming pipeline (it's pre-recorded) — using the *upcoming* frames' saliency/motion as an additional input, not just past ones, is a realistic enhancement most FoV-prediction systems exploit and this notebook doesn't yet use.
